# Electronic Arts - Game Library Health for User Retention

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from datetime import date

In [3]:
df_dim = pd.read_csv('../Data/023/dim_users.csv', parse_dates=['library_last_updated'])
df_fct_user = pd.read_csv('../Data/023/fct_user_game_activity.csv', parse_dates=['install_date','last_played_date'])

pl_dim = pl.read_csv('../Data/023/dim_users.csv').with_columns(pl.col('library_last_updated').str.to_date('%Y-%m-%d'))
pl_fct_user = pl.read_csv('../Data/023/fct_user_game_activity.csv').with_columns(pl.col(['install_date','last_played_date']).str.to_date('%Y-%m-%d'))


# Pregunta 1

### ¿Cuántos usuarios no tienen juegos instalados en su biblioteca durante el tercer trimestre de 2024? Utiliza la tabla dim_users y filtra por los usuarios donde has_game_installed es 0 y la fecha library_last_updated esté entre julio y septiembre de 2024. Esto ayuda a identificar a los usuarios a los que se puede dirigir una campaña para aumentar el compromiso (engagement).

```SQL
SELECT
    COUNT(*)
FROM dim_users
WHERE (library_last_updated BETWEEN '2024-07-01' AND '2024-09-30') AND
      has_game_installed = FALSE;
```

In [4]:
res = df_dim[
    (df_dim['library_last_updated'].between('2024-07-01','2024-09-30')) &
    (df_dim['has_game_installed'] == False)
].shape[0]

res

14

In [ ]:
res = pl_dim.filter(
    (pl.col('library_last_updated').is_between(date(2024,7,1),date(2024,9,30))) &
    (pl.col('has_game_installed') == False)
).height

res

14

# Pregunta 2

### ¿Cuáles fueron los 5 juegos que tuvieron el mayor número de instalaciones durante el tercer trimestre de 2024? Esto ayuda a revelar los juegos más populares entre los usuarios.

```SQL
SELECT
    game_id,
    COUNT(install_date) AS num_installed
FROM fct_user_game_activity
WHERE (install_date BETWEEN '2024-07-01' AND '2024-09-30')
GROUP BY game_id
ORDER BY num_installed DESC
LIMIT 5;
```

In [11]:
df_q3 = df_fct_user[
    (df_fct_user['install_date'].between('2024-07-01','2024-09-30'))
].reset_index()

res = df_q3.groupby('game_id')['install_date'].count().reset_index(name='num_installed')

res = res.sort_values(by='num_installed', ascending=False).head(5)

In [13]:
res = pl_fct_user.filter(
    (pl.col('install_date').is_between(date(2024,7,1),date(2024,9,30)))
).group_by('game_id').agg(
    pl.col('install_date').count().alias('num_installed')
).sort('num_installed', descending=True).head(5)

# Pregunta 3

### ¿Cuántos usuarios, cuyas bibliotecas fueron actualizadas por última vez entre julio y septiembre de 2024, tienen 3 o más juegos instalados en su biblioteca?

```SQL
SELECT
    COUNT(DISTINCT user_id)
FROM dim_users
WHERE (library_last_updated BETWEEN '2024-07-01' AND '2024-09-30') AND
      installed_games_count >= 3
```

In [14]:
df_q3 = df_dim[
    (df_dim['library_last_updated'].between('2024-07-01','2024-09-30')) &
    (df_dim['installed_games_count'] >= 3)
].reset_index()

res = df_q3['user_id'].nunique()

res

7

In [15]:
res = pl_dim.filter(
    (pl.col('library_last_updated').is_between(date(2024,7,1),date(2024,9,30))) &
    (pl.col('installed_games_count') >= 3)
).select(
    pl.col('user_id').n_unique()
)

res

user_id
u32
7
